# 4-Speaker Speech Separation — SepFormer 3→4 transfer
**How to run (Kaggle):** Settings → Accelerator → **GPU T4 x2** → run cells top-to-bottom.
For the overnight run: Save Version → **Save & Run All (Commit)**. Total runtime ≈ 6–9 h.

**Outputs (in the version's Output tab):** `sepformer4.ckpt` (fine-tuned model),
`demo_mix4.wav` + `demo_cascade_out*.wav` + `demo_finetuned_out*.wav` (listening demos),
and the log contains: cascade SI-SDRi, training curve, final fine-tuned SI-SDRi — both models
scored on the SAME 10 fixed test mixes.

In [ ]:
# Cell 1 — environment + hard verification (wrong accelerator dies here, not 6 cells later)
!pip -q install speechbrain asteroid torchaudio soundfile
import torch, torchaudio, glob, random, os
print(torch.__version__, "| CUDA:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))
assert torch.cuda.get_device_capability(0) >= (7, 0), "Old GPU (P100) — switch Settings to GPU T4 x2"

In [ ]:
# Cell 2 — speech pool: LibriSpeech dev-clean (~40 speakers). We manufacture mixtures
# ourselves because mixing is addition — so we always own the ground truth.
if not os.path.exists("LibriSpeech/dev-clean"):
    !wget -q https://www.openslr.org/resources/12/dev-clean.tar.gz
    !tar -xzf dev-clean.tar.gz
flacs = glob.glob("LibriSpeech/dev-clean/**/*.flac", recursive=True)
by_spk = {}
for f in flacs:
    by_spk.setdefault(f.split("/")[-3], []).append(f)
print(len(flacs), "files |", len(by_spk), "speakers")
assert len(by_spk) >= 20, "dataset download failed"

In [ ]:
# Cell 3 — mixture factory. SR=8000 because BOTH pretrained SepFormers are 8 kHz models.
# On clipping we rescale mix AND sources by the same factor, so mix == sum(sources) stays true.
SR = 8000

def load_8k(path, seg=None):
    wav, sr = torchaudio.load(path)
    wav = torchaudio.functional.resample(wav, sr, SR).mean(0)
    if seg:
        if wav.shape[0] < seg:
            wav = torch.nn.functional.pad(wav, (0, seg - wav.shape[0]))
        else:
            s = random.randint(0, wav.shape[0] - seg)
            wav = wav[s:s+seg]
    return wav

def make_mix(n_spk=4, seg=SR*3):
    spks = random.sample(list(by_spk), n_spk)
    srcs = []
    for s in spks:
        w = load_8k(random.choice(by_spk[s]), seg)
        w = w / (w.abs().max() + 1e-8) * random.uniform(0.5, 1.0)
        srcs.append(w)
    srcs = torch.stack(srcs)
    mix = srcs.sum(0)
    scale = 0.9 / max(mix.abs().max().item(), 1e-8)
    if scale < 1:
        mix, srcs = mix * scale, srcs * scale
    return mix, srcs

In [ ]:
# Cell 4 — pretrained 3-speaker baseline + demo audio (proves the whole stack works)
from speechbrain.inference.separation import SepformerSeparation as Sep
DEV = {"device": "cuda:0"}
m3 = Sep.from_hparams("speechbrain/sepformer-wsj03mix", savedir="sf3", run_opts=DEV)

mix, srcs = make_mix(3, seg=SR*4)
est = m3.separate_batch(mix.unsqueeze(0).cuda())
torchaudio.save("demo_mix3.wav", mix.unsqueeze(0), SR)
for i in range(3):
    torchaudio.save(f"demo_3spk_out{i+1}.wav", est[0, :, i].unsqueeze(0).cpu(), SR)
print("3-speaker demo saved")

In [ ]:
# Cell 5 — zero-training 4-speaker CASCADE: 3-mix model makes 3 streams; the 2-mix model
# then splits whichever stream still holds two voices. Scoring: a true 2-speaker stream
# splits into two DISSIMILAR (low cosine), comparable-energy outputs; a 1-speaker stream
# yields near-copies or one voice + low-energy junk.
m2 = Sep.from_hparams("speechbrain/sepformer-wsj02mix", savedir="sf2", run_opts=DEV)

def split_score(a, b):
    cos = torch.nn.functional.cosine_similarity(a, b, dim=0).abs().item()
    ra, rb = a.pow(2).mean().sqrt().item(), b.pow(2).mean().sqrt().item()
    return (1 - cos) * (min(ra, rb) / (max(ra, rb) + 1e-8))

def separate4_cascade(mix):                                # mix: (T,) on cuda
    tri = m3.separate_batch(mix.unsqueeze(0))[0]           # (T, 3)
    cands = []
    for i in range(3):
        duo = m2.separate_batch(tri[:, i].unsqueeze(0))[0] # (T, 2)
        cands.append((split_score(duo[:, 0], duo[:, 1]), i, duo))
    _, k, duo = max(cands, key=lambda c: c[0])
    keep = [tri[:, i] for i in range(3) if i != k]
    L = min(min(t.shape[0] for t in keep), duo.shape[0])
    return torch.stack([t[:L] for t in keep] + [duo[:L, 0], duo[:L, 1]])

In [ ]:
# Cell 6 — SI-SDRi metric (PIT-matched) + a FIXED 10-mix test set reused for every model,
# so cascade vs fine-tuned is an apples-to-apples comparison.
from asteroid.losses import PITLossWrapper, pairwise_neg_sisdr
pit = PITLossWrapper(pairwise_neg_sisdr, pit_from="pw_mtx")

def sisdri(est, srcs, mix):
    est = est.detach().cpu()
    L = min(est.shape[1], srcs.shape[1], mix.shape[0])
    est, srcs, mix = est[:, :L], srcs[:, :L], mix[:L]
    best = -pit(est.unsqueeze(0), srcs.unsqueeze(0)).item()
    base = -pit(mix.repeat(len(srcs), 1).unsqueeze(0), srcs.unsqueeze(0)).item()
    return best - base

random.seed(7)
test_set = [make_mix(4, seg=SR*4) for _ in range(10)]

cascade_scores = [sisdri(separate4_cascade(m.cuda()), s, m) for m, s in test_set]
CASCADE = sum(cascade_scores) / len(cascade_scores)
print("CASCADE SI-SDRi:", round(CASCADE, 2), "dB")

m0, s0 = test_set[0]
out = separate4_cascade(m0.cuda()).cpu()
torchaudio.save("demo_mix4.wav", m0.unsqueeze(0), SR)
for i in range(4):
    torchaudio.save(f"demo_cascade_out{i+1}.wav", out[i].unsqueeze(0), SR)

In [ ]:
# Cell 7 — SURGERY: the transformer body learned WHAT SPEECH LOOKS LIKE (count-agnostic);
# only conv2d (256 -> 256*num_spks) deals features into per-speaker piles. We widen it 3->4,
# copying the 3 pretrained dealers so only dealer 4 starts from scratch.
import torch.nn as nn
m4 = Sep.from_hparams("speechbrain/sepformer-wsj03mix", savedir="sf4", run_opts=DEV)
mn = m4.mods.masknet
print("num_spks before:", mn.num_spks)

old = mn.conv2d
new = nn.Conv2d(256, 256*4, kernel_size=1)
with torch.no_grad():
    new.weight[:768] = old.weight
    new.bias[:768]   = old.bias
mn.conv2d = new.cuda()
mn.num_spks = 4

masks = mn(m4.mods.encoder(torch.randn(1, SR*3).cuda()))
print("masks:", tuple(masks.shape))          # expect (4, 1, 256, L)
assert masks.shape[0] == 4

In [ ]:
# Cell 8 — overnight PIT fine-tune. Forward in fp16 (fits T4), loss in fp32 (SI-SNR's
# ratios/logs are unstable in half precision). PIT = try all 4! pairings, backprop the best.
enc, mn, dec = m4.mods.encoder, m4.mods.masknet, m4.mods.decoder
m4.mods.train()
opt = torch.optim.Adam(m4.mods.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler("cuda")

def forward4(mix):                               # (B, T) -> (B, 4, T')
    w = enc(mix)
    masks = mn(w)
    return torch.stack([dec(w * masks[i]) for i in range(4)], dim=1)

for step in range(1, 20001):
    mix, srcs = make_mix(4)
    mix, srcs = mix.unsqueeze(0).cuda(), srcs.unsqueeze(0).cuda()
    with torch.amp.autocast("cuda"):
        est = forward4(mix)
    L = min(est.shape[-1], srcs.shape[-1])
    loss = pit(est[..., :L].float(), srcs[..., :L])
    opt.zero_grad(); scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(m4.mods.parameters(), 5)
    scaler.step(opt); scaler.update()
    if step == 10000:
        for g in opt.param_groups: g["lr"] = 5e-5
    if step % 100 == 0:
        print(step, "train SI-SNR:", round(-loss.item(), 2), "dB", flush=True)
    if step % 1000 == 0:
        torch.save(m4.mods.state_dict(), "sepformer4.ckpt")

In [ ]:
# Cell 9 — final head-to-head on the SAME fixed test set + demo wavs + final checkpoint.
# NOTE: never use m4.separate_batch() after surgery (its config still says 3 speakers) —
# always go through forward4.
m4.mods.eval()

def separate4_ft(mix):                           # (T,) -> (4, T)
    with torch.no_grad():
        return forward4(mix.unsqueeze(0).cuda())[0]

ft_scores = [sisdri(separate4_ft(m), s, m) for m, s in test_set]
FT = sum(ft_scores) / len(ft_scores)
print("FINE-TUNED SI-SDRi:", round(FT, 2), "dB")
print("CASCADE    SI-SDRi:", round(CASCADE, 2), "dB")

out = separate4_ft(test_set[0][0]).cpu()
for i in range(4):
    torchaudio.save(f"demo_finetuned_out{i+1}.wav", out[i].unsqueeze(0), SR)
torch.save(m4.mods.state_dict(), "sepformer4.ckpt")
print("done — checkpoint + demo wavs are in the Output tab")